In [1]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data
!unzip -q /content/drive/MyDrive/major_project/vctk_16k.zip -d /content/data

import glob
for root in glob.glob('/content/data/**/*_wav', recursive=True):
    n = len(glob.glob(root + '/*.wav'))
    print(root, n)

Mounted at /content/drive
/content/data/vctk_16k/noisy_testset_wav 824
/content/data/vctk_16k/clean_testset_wav 824
/content/data/vctk_16k/noisy_trainset_28spk_wav 11572
/content/data/vctk_16k/clean_trainset_28spk_wav 11572


In [2]:
!pip install -q pesq pystoi soundfile
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

  Preparing metadata (setup.py) ... done
GPU available: True
GPU name: Tesla T4


In [3]:
import os, random, numpy as np, soundfile as sf
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = '/content/data/vctk_16k'
SEG, N_FFT, HOP = 32000, 512, 128      # 2-second crops at 16 kHz
device = 'cuda'

class VCTKPairs(Dataset):
    def __init__(self, split, train=True):
        self.cdir = f'{ROOT}/clean_{split}'
        self.ndir = f'{ROOT}/noisy_{split}'
        self.files = sorted(os.listdir(self.cdir))
        self.train = train
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        name = self.files[i]
        c, _ = sf.read(f'{self.cdir}/{name}', dtype='float32')
        n, _ = sf.read(f'{self.ndir}/{name}', dtype='float32')
        L = min(len(c), len(n)); c, n = c[:L], n[:L]
        if L >= SEG:
            s = random.randint(0, L - SEG) if self.train else 0
            c, n = c[s:s+SEG], n[s:s+SEG]
        else:
            c = np.pad(c, (0, SEG - L)); n = np.pad(n, (0, SEG - L))
        return torch.from_numpy(n), torch.from_numpy(c)

def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, b=16):
        super().__init__()
        self.e1, self.e2, self.e3 = block(1, b), block(b, 2*b), block(2*b, 4*b)
        self.bott = block(4*b, 8*b)
        self.u3 = nn.ConvTranspose2d(8*b, 4*b, 2, stride=2); self.d3 = block(8*b, 4*b)
        self.u2 = nn.ConvTranspose2d(4*b, 2*b, 2, stride=2); self.d2 = block(4*b, 2*b)
        self.u1 = nn.ConvTranspose2d(2*b, b, 2, stride=2);   self.d1 = block(2*b, b)
        self.out = nn.Conv2d(b, 1, 1)
    def forward(self, x):                      # x: (B, 1, freq, time)
        T = x.shape[-1]
        x = F.pad(x, (0, (-T) % 8))
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        m = self.bott(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(m), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))[..., :T]

window = torch.hann_window(N_FFT, device=device)
def stft(w):
    return torch.stft(w, N_FFT, HOP, window=window, return_complex=True)

# ---- quick test on one batch ----
train_ds = VCTKPairs('trainset_28spk_wav', train=True)
loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
model = UNet().to(device)
print('Model parameters (millions):', sum(p.numel() for p in model.parameters()) / 1e6)

noisy, clean = next(iter(loader))
noisy, clean = noisy.to(device), clean.to(device)
mag = stft(noisy).abs()[:, :256]               # drop top bin so freq = 256
mask = model(torch.log1p(mag).unsqueeze(1))
print('waveform:', tuple(noisy.shape), '| spectrogram:', tuple(mag.shape), '| mask:', tuple(mask.shape))

Model parameters (millions): 0.483153
waveform: (8, 32000) | spectrogram: (8, 256, 251) | mask: (8, 1, 256, 251)


In [4]:
import time
EPOCHS = 3
BATCH = 16
SAVE = '/content/drive/MyDrive/major_project/enh_unet.pth'

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
val_ds = VCTKPairs('testset_wav', train=False)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)

model = UNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_batch(noisy, clean):
    noisy, clean = noisy.to(device), clean.to(device)
    nmag = stft(noisy).abs()[:, :256]
    cmag = stft(clean).abs()[:, :256]
    mask = model(torch.log1p(nmag).unsqueeze(1)).squeeze(1)
    # L1 loss on log-compressed magnitude: masked noisy vs clean
    return F.l1_loss(torch.log1p(mask * nmag), torch.log1p(cmag))

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train(); tr = 0
    for noisy, clean in train_loader:
        loss = run_batch(noisy, clean)
        opt.zero_grad(); loss.backward(); opt.step()
        tr += loss.item()
    model.eval(); va = 0
    with torch.no_grad():
        for noisy, clean in val_loader:
            va += run_batch(noisy, clean).item()
    print(f'epoch {ep}/{EPOCHS} | train loss {tr/len(train_loader):.4f} | '
          f'val loss {va/len(val_loader):.4f} | {time.time()-t0:.0f}s')
    torch.save(model.state_dict(), SAVE)
print('saved to', SAVE)

epoch 1/3 | train loss 0.0329 | val loss 0.0165 | 130s
epoch 2/3 | train loss 0.0278 | val loss 0.0148 | 131s
epoch 3/3 | train loss 0.0264 | val loss 0.0148 | 132s
saved to /content/drive/MyDrive/major_project/enh_unet.pth


In [5]:
from pesq import pesq
from pystoi import stoi

def si_snr(est, ref):
    est, ref = est - est.mean(), ref - ref.mean()
    proj = (est * ref).sum() / ((ref * ref).sum() + 1e-8) * ref
    noise = est - proj
    return 10 * np.log10(((proj**2).sum() + 1e-8) / ((noise**2).sum() + 1e-8))

def enhance(n):
    w = torch.from_numpy(n).to(device)
    S = stft(w.unsqueeze(0))
    mag = S.abs()
    with torch.no_grad():
        m = model(torch.log1p(mag[:, :256]).unsqueeze(1)).squeeze(1)
    m = torch.cat([m, m[:, -1:]], 1)
    out = torch.istft(S * m, N_FFT, HOP, window=window, length=len(n))
    return out.squeeze(0).cpu().numpy()

model.eval()
cdir, ndir = f'{ROOT}/clean_testset_wav', f'{ROOT}/noisy_testset_wav'
files = sorted(os.listdir(cdir))[:100]
out_dir = '/content/drive/MyDrive/major_project/samples'
os.makedirs(out_dir, exist_ok=True)

res = {'noisy': [], 'enhanced': []}
for k, f in enumerate(files):
    c, _ = sf.read(f'{cdir}/{f}', dtype='float32')
    n, _ = sf.read(f'{ndir}/{f}', dtype='float32')
    L = min(len(c), len(n)); c, n = c[:L], n[:L]
    e = enhance(n)
    for name, x in [('noisy', n), ('enhanced', e)]:
        try:
            p = pesq(16000, c, x, 'wb')
        except Exception:
            p = np.nan
        res[name].append((p, stoi(c, x, 16000), si_snr(x, c)))
    if k < 5:
        sf.write(f'{out_dir}/{f[:-4]}_noisy.wav', n, 16000)
        sf.write(f'{out_dir}/{f[:-4]}_enhanced.wav', e, 16000)

print(f'{"":10s} {"PESQ":>7s} {"STOI":>7s} {"SI-SNR":>8s}')
for name in res:
    a = np.array(res[name])
    print(f'{name:10s} {np.nanmean(a[:,0]):7.3f} {a[:,1].mean():7.3f} {a[:,2].mean():8.2f}')

              PESQ    STOI   SI-SNR
noisy        2.162   0.920     8.50
enhanced     2.696   0.935    18.35


In [6]:
EPOCHS = 25
BEST = '/content/drive/MyDrive/major_project/enh_unet_best.pth'

model.load_state_dict(torch.load(SAVE))      # continue from the 3-epoch model
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
best = 1e9

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train(); tr = 0
    for noisy, clean in train_loader:
        loss = run_batch(noisy, clean)
        opt.zero_grad(); loss.backward(); opt.step()
        tr += loss.item()
    sched.step()
    model.eval(); va = 0
    with torch.no_grad():
        for noisy, clean in val_loader:
            va += run_batch(noisy, clean).item()
    va /= len(val_loader)
    if va < best:
        best = va
        torch.save(model.state_dict(), BEST)
    print(f'epoch {ep}/{EPOCHS} | train {tr/len(train_loader):.4f} | '
          f'val {va:.4f} | best {best:.4f} | {time.time()-t0:.0f}s')
print('best model saved to', BEST)

epoch 1/25 | train 0.0259 | val 0.0150 | best 0.0150 | 132s
epoch 2/25 | train 0.0252 | val 0.0147 | best 0.0147 | 132s
epoch 3/25 | train 0.0248 | val 0.0144 | best 0.0144 | 132s
epoch 4/25 | train 0.0243 | val 0.0144 | best 0.0144 | 132s
epoch 5/25 | train 0.0241 | val 0.0146 | best 0.0144 | 132s
epoch 6/25 | train 0.0239 | val 0.0155 | best 0.0144 | 132s
epoch 7/25 | train 0.0236 | val 0.0144 | best 0.0144 | 132s
epoch 8/25 | train 0.0234 | val 0.0143 | best 0.0143 | 132s
epoch 9/25 | train 0.0233 | val 0.0142 | best 0.0142 | 132s
epoch 10/25 | train 0.0230 | val 0.0143 | best 0.0142 | 132s
epoch 11/25 | train 0.0229 | val 0.0143 | best 0.0142 | 132s
epoch 12/25 | train 0.0227 | val 0.0140 | best 0.0140 | 132s
epoch 13/25 | train 0.0226 | val 0.0138 | best 0.0138 | 132s
epoch 14/25 | train 0.0225 | val 0.0139 | best 0.0138 | 132s
epoch 15/25 | train 0.0223 | val 0.0140 | best 0.0138 | 132s
epoch 16/25 | train 0.0222 | val 0.0139 | best 0.0138 | 132s
epoch 17/25 | train 0.0221 | val 

In [7]:
import pandas as pd

model.load_state_dict(torch.load('/content/drive/MyDrive/major_project/enh_unet_best.pth'))
model.eval()

files = sorted(os.listdir(cdir))          # all 824 test clips
rows = []
for k, f in enumerate(files):
    c, _ = sf.read(f'{cdir}/{f}', dtype='float32')
    n, _ = sf.read(f'{ndir}/{f}', dtype='float32')
    L = min(len(c), len(n)); c, n = c[:L], n[:L]
    e = enhance(n)
    row = {'file': f}
    for name, x in [('noisy', n), ('enhanced', e)]:
        try:
            p = pesq(16000, c, x, 'wb')
        except Exception:
            p = np.nan
        row[f'{name}_pesq'] = p
        row[f'{name}_stoi'] = stoi(c, x, 16000)
        row[f'{name}_sisnr'] = si_snr(x, c)
    rows.append(row)
    if 5 <= k < 15:                       # 10 more before/after pairs
        sf.write(f'{out_dir}/{f[:-4]}_noisy.wav', n, 16000)
        sf.write(f'{out_dir}/{f[:-4]}_enhanced.wav', e, 16000)

df = pd.DataFrame(rows)
df.to_csv('/content/drive/MyDrive/major_project/final_eval_per_file.csv', index=False)

print(f'Evaluated {len(df)} files')
print(f'{"":10s} {"PESQ":>7s} {"STOI":>7s} {"SI-SNR":>8s}')
for name in ['noisy', 'enhanced']:
    print(f'{name:10s} {df[name+"_pesq"].mean():7.3f} {df[name+"_stoi"].mean():7.3f} {df[name+"_sisnr"].mean():8.2f}')

Evaluated 824 files
              PESQ    STOI   SI-SNR
noisy        1.972   0.921     8.44
enhanced     2.806   0.941    18.18
